[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Update Operators


## What you will be able to do

Change documents without losing the fields you did not mention. Say what `update_one` and
`replace_one` each do, why each refuses the other's argument, and which one deletes half your
document. Use `$set`, `$unset`, `$inc`, `$push`, `$addToSet`, the positional `$` and `array_filters`.
Say what `upsert=True` inserts when the filter matches nothing, which is not always what you would
guess. And know which direction `find_one_and_update` looks by default.


## The idea

### The problem

There are two ways to change a document and they look alike from a distance. One takes a description
of the changes. The other takes a whole new document. Pass the wrong shape and PyMongo catches it,
which is good; pass the right shape to the wrong method and it works perfectly and destroys
everything you left out.

### What an update document is

A dictionary whose keys are operators: `{"$set": {...}, "$inc": {...}}`. Every key must begin with
`$`, and each operator names the fields it touches. Fields not named are not touched.

### What a replacement is

A whole document, with no operators in it at all. It becomes the document, keeping only the `_id`.
Anything absent from it is gone.

### Why upsert can surprise you

When the filter matches nothing, MongoDB builds the new document out of **the filter's equality
conditions plus the update**. So the fields you filtered on end up in the inserted document, and a
filter with an extra condition inserts a different document than you expected.

### Where this shows up

Every write after the first. The `replace_one` trap in particular is how a schema change loses data:
somebody reads a document, adds a field in Python, and writes it back with `replace_one`, and every
field added by another process since the read is gone.

### What this notebook covers

`update_one` and `update_many`, `replace_one`, and the two `ValueError`s that keep them apart.
`$set`, `$unset`, `$inc`, `$mul`, `$rename`. Arrays: `$push`, `$addToSet`, `$pull`, the positional
`$` and `array_filters`. `upsert`. `find_one_and_update` and `ReturnDocument`. Then the four
failures.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

shop.items.drop()
shop.items.insert_one({"_id": 1, "name": "keyboard", "price": 50, "stock": 4, "tags": ["sale"]})

shop.items.update_one({"_id": 1}, {"$set": {"price": 60}})
print("after $set:       ", sorted(shop.items.find_one({"_id": 1})))

shop.items.replace_one({"_id": 1}, {"name": "keyboard", "price": 70})
print("after replace_one:", sorted(shop.items.find_one({"_id": 1})))
print("stock and tags are gone, and nothing said so")
client.close()
```

```
after $set:        ['_id', 'name', 'price', 'stock', 'tags']
after replace_one: ['_id', 'name', 'price']
stock and tags are gone, and nothing said so
```

Both calls succeeded. Both reported one document modified. One of them changed a price and the other
changed a price and deleted two fields, and the only visible difference is which method was called.


## Setup

Eight imports, MongoDB, the boot cell, and three helpers.

- `pymongo` is the driver and `ReturnDocument` names which side of a change you want back
- `subprocess` and `os` install and start the server, `sys` names this Python, `time` waits
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

`fresh_item` puts one document back exactly as it was, so each section below starts from the same
place and the notebook can be run from the top as often as you like. `item` reads it. `failed`
prints a write failure's message without the parts that differ between runs.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo
from pymongo import ReturnDocument

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """A write or command failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    return f"{type(error).__name__}: {details.get('errmsg', str(error).split(', full error')[0])}"


def fresh_item():
    """One document to write on, put back exactly as it was before each section."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.items.delete_many({"_id": 1})
        shop.items.insert_one({
            "_id": 1,
            "name": "keyboard",
            "price": 50,
            "stock": 4,
            "tags": ["sale"],
            "lines": [{"sku": "a", "qty": 1}, {"sku": "b", "qty": 2}],
        })
        return shop.items.find_one({"_id": 1})


def item():
    """What is in it now."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        return client.get_default_database().items.find_one({"_id": 1})


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print("item:   ", sorted(fresh_item()))
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
item:    ['_id', 'lines', 'name', 'price', 'stock', 'tags']
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### The two methods, and the two refusals

PyMongo checks the shape before the server sees it:


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
fresh_item()

for label, call in (("update_one with a plain document",
                     lambda: shop.items.update_one({"_id": 1}, {"price": 60})),
                    ("replace_one with operators",
                     lambda: shop.items.replace_one({"_id": 1}, {"$set": {"price": 60}}))):
    try:
        call()
    except ValueError as error:
        print(f"  {label:34} -> ValueError: {error}")


  update_one with a plain document   -> ValueError: update only works with $ operators
  replace_one with operators         -> ValueError: replacement can not include $ operators


Two errors, raised in Python, before a byte crosses the network. They are the most useful errors in
this notebook because they are the only ones that stop you doing the wrong thing.

The dangerous combination is the one that is not an error: a whole document handed to `replace_one`,
which is exactly what it wants.

### $set, $unset and the fields you did not mention


In [3]:
fresh_item()
print("before:", sorted(item()))

shop.items.update_one({"_id": 1}, {"$set": {"price": 60, "colour": "black"}})
print("after $set:", sorted(item()), "<- colour was added, nothing was lost")

shop.items.update_one({"_id": 1}, {"$unset": {"colour": ""}})
print("after $unset:", sorted(item()), "<- and removed again")


before: ['_id', 'lines', 'name', 'price', 'stock', 'tags']
after $set: ['_id', 'colour', 'lines', 'name', 'price', 'stock', 'tags'] <- colour was added, nothing was lost
after $unset: ['_id', 'lines', 'name', 'price', 'stock', 'tags'] <- and removed again


`$unset` takes a value and ignores it; `""` is the convention. `$set` creates a field that was not
there, which is how a collection grows a new field without a migration.

### Numbers

`$inc` changes a number by an amount, on the server:


In [4]:
fresh_item()
shop.items.update_one({"_id": 1}, {"$inc": {"stock": -1}})
print("after selling one:", item()["stock"])

shop.items.update_one({"_id": 1}, {"$mul": {"price": 2}})
print("after doubling:   ", item()["price"])

shop.items.update_one({"_id": 1}, {"$rename": {"colour": "color"}})
print("renaming a field that is not there is not an error:", sorted(item()))


after selling one: 3
after doubling:    100
renaming a field that is not there is not an error: ['_id', 'lines', 'name', 'price', 'stock', 'tags']


`$inc` matters more than it looks. Reading a number, adding one in Python and writing it back is two
round trips with a gap in the middle, and two processes doing it at once lose one of the increments.
`$inc` is one operation on the server and cannot be interleaved.

### Arrays


In [5]:
fresh_item()
shop.items.update_one({"_id": 1}, {"$push": {"tags": "new"}})
shop.items.update_one({"_id": 1}, {"$push": {"tags": "sale"}})
print("after two $push:    ", item()["tags"], "<- sale is in there twice")

fresh_item()
shop.items.update_one({"_id": 1}, {"$addToSet": {"tags": "new"}})
shop.items.update_one({"_id": 1}, {"$addToSet": {"tags": "sale"}})
print("after two $addToSet:", item()["tags"], "<- sale was already there")

shop.items.update_one({"_id": 1}, {"$pull": {"tags": "new"}})
print("after $pull:        ", item()["tags"])


after two $push:     ['sale', 'new', 'sale'] <- sale is in there twice
after two $addToSet: ['sale', 'new'] <- sale was already there
after $pull:         ['sale']


`$push` appends whatever it is given. `$addToSet` appends only if the value is not already present,
which is the array equivalent of a set and the one to reach for when duplicates would be wrong.

To change one element of an array, the positional `$` stands for the element the filter matched:


In [6]:
fresh_item()
print("before:", item()["lines"])

shop.items.update_one({"_id": 1, "lines.sku": "b"}, {"$set": {"lines.$.qty": 20}})
print("after positional $:", item()["lines"])


before: [{'sku': 'a', 'qty': 1}, {'sku': 'b', 'qty': 2}]
after positional $: [{'sku': 'a', 'qty': 1}, {'sku': 'b', 'qty': 20}]


The `$` works only when the filter matched exactly the element you mean, and it changes the **first**
matching element and no others. For anything more complicated, name the element with
`array_filters`:


In [7]:
fresh_item()
shop.items.update_one({"_id": 1},
                      {"$set": {"lines.$[small].qty": 99}},
                      array_filters=[{"small.qty": {"$lt": 2}}])
print("every line with qty under 2:", item()["lines"])


every line with qty under 2: [{'sku': 'a', 'qty': 99}, {'sku': 'b', 'qty': 2}]


`small` is an identifier you invent. It appears in the path as `$[small]` and in `array_filters` as
the prefix of every condition, and the two must agree, which is the subject of one of the errors
below.

### upsert, and what it builds

When the filter matches nothing, the new document is made of the filter's equality conditions plus
the update:


In [8]:
shop.stock.drop()
outcome = shop.stock.update_one({"sku": "NEW-1"}, {"$set": {"count": 5}}, upsert=True)

print("matched:", outcome.matched_count, "| modified:", outcome.modified_count,
      "| upserted an id:", outcome.upserted_id is not None)
print("the document it built:", shop.stock.find_one({"sku": "NEW-1"}, {"_id": 0}))
print("sku came from the filter and count came from the update")


matched: 0 | modified: 0 | upserted an id: True
the document it built: {'sku': 'NEW-1', 'count': 5}
sku came from the filter and count came from the update


Only equality conditions contribute. A filter with `{"count": {"$gt": 0}}` in it puts nothing in the
new document, because `$gt` is not a value.

### find_one_and_update, and which side it returns


In [9]:
fresh_item()

before = shop.items.find_one_and_update({"_id": 1}, {"$inc": {"stock": 1}})
print("default:              stock came back as", before["stock"], "and is now", item()["stock"])

after = shop.items.find_one_and_update({"_id": 1}, {"$inc": {"stock": 1}},
                                       return_document=ReturnDocument.AFTER)
print("ReturnDocument.AFTER: stock came back as", after["stock"])


default:              stock came back as 4 and is now 5
ReturnDocument.AFTER: stock came back as 6


The default is `BEFORE`, which is the opposite of what most people expect and is the right default
for a queue: you take the document as it was and mark it taken, in one operation, and no other
process can take it between the two.

### When to reach for which

| What you want | How to write it |
|---|---|
| change some fields | `update_one(filter, {"$set": {...}})` |
| remove a field | `{"$unset": {"field": ""}}` |
| add to a number safely | `{"$inc": {"stock": -1}}` |
| replace the whole document | `replace_one(filter, document)`, deliberately |
| append to an array | `{"$push": {"tags": "new"}}` |
| append only if absent | `{"$addToSet": {"tags": "new"}}` |
| remove from an array | `{"$pull": {"tags": "new"}}` |
| change the matched element | `{"$set": {"lines.$.qty": 1}}` |
| change several elements | `array_filters=[{"x.qty": {"$lt": 2}}]` |
| insert when nothing matches | `upsert=True` |
| read and change atomically | `find_one_and_update(..., return_document=...)` |
| change every match | `update_many` |

The default is `update_one` with `$set`. Reach for `replace_one` only when you really do mean "this
document is now exactly this", and when you do, make sure you built the replacement from a document
you just read rather than from a partial dictionary.

### Taking one item from stock, atomically, finished


In [10]:
def take_one(shop, sku):
    """Decrement stock and hand back the item as it was, or None if there was none left."""
    return shop.stock.find_one_and_update(
        {"sku": sku, "count": {"$gt": 0}},                          # only if there is any
        {"$inc": {"count": -1}},
        return_document=ReturnDocument.AFTER,                       # what is left
    )


shop.stock.drop()
shop.stock.insert_one({"sku": "KEY-1", "count": 2})

for attempt in range(1, 4):
    left = take_one(shop, "KEY-1")
    print(f"  attempt {attempt}: {'took one, ' + str(left['count']) + ' left' if left else 'none left'}")

print()
print("the filter and the change are one operation, so two processes cannot both take the last one")


  attempt 1: took one, 1 left
  attempt 2: took one, 0 left
  attempt 3: none left

the filter and the change are one operation, so two processes cannot both take the last one


The condition `{"count": {"$gt": 0}}` is inside the same operation as the decrement, which is what
makes this safe. Reading the count first and then decrementing would be two operations with a gap,
and the gap is where the count goes negative.

`None` is how the function says "nothing matched", which here means "nothing left", because the only
way to match nothing is for the count to have hit zero.

### Where each part came from

| In `take_one` | What it relies on | The section that showed it |
|---|---|---|
| `find_one_and_update` | reading and changing in one operation | find_one_and_update |
| `return_document=AFTER` | the default being `BEFORE` | find_one_and_update |
| `{"$inc": {"count": -1}}` | arithmetic on the server | Numbers |
| `{"count": {"$gt": 0}}` in the filter | an operator inside its field | **Query Operators** |
| `None` when nothing matched | `find_one_and_update` returning `None` | find_one_and_update |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/06-update-operators-solutions.ipynb).

**1.** Change one field with `$set` and show the others survived.


In [11]:
# your code here


**2.** Replace the same document with `replace_one` and show what is left.


In [12]:
# your code here


**3.** Reduce `stock` by one without reading it first.


In [13]:
# your code here


**4.** Add a tag twice with `$push`, then twice with `$addToSet`.


In [14]:
# your code here


**5.** Upsert a document that does not exist and print what got built.


In [15]:
# your code here


**6.** Show both directions of `find_one_and_update`.


In [16]:
# your code here


## Common errors

### ValueError: update only works with $ operators


In [17]:
shop.items.update_one({"_id": 1}, {"price": 60})


ValueError: update only works with $ operators

Raised by PyMongo, not by the server. It is checking that you did not mean `replace_one`, and it is
right to: `{"price": 60}` as an update is meaningless, but as a replacement it is a complete
document that would have deleted everything else.

Its mirror refuses the opposite mistake:


In [18]:
try:
    shop.items.replace_one({"_id": 1}, {"$set": {"price": 60}})
except ValueError as error:
    print("ValueError:", error)

print()
print("the pair of them is the only thing standing between you and a silent data loss")


ValueError: replacement can not include $ operators

the pair of them is the only thing standing between you and a silent data loss


### pymongo.errors.WriteError: Unknown modifier: $addtoset


In [19]:
try:
    shop.items.update_one({"_id": 1}, {"$addtoset": {"tags": "new"}})
except pymongo.errors.WriteError as error:
    print(failed(error).split(". Expected")[0])


WriteError: Unknown modifier: $addtoset


Operators are case sensitive and this one has three capitals in the middle of it. `$addToSet`,
`$setOnInsert`, `$currentDate` and `$bit` are the ones that catch people, and the error is at least
specific about which modifier it did not recognize.

There is no way to make this a compile-time error in Python, so the defense is to keep operator
names in one place if you build updates dynamically:


In [20]:
fresh_item()
shop.items.update_one({"_id": 1}, {"$addToSet": {"tags": "new"}})
print("with the right spelling:", item()["tags"])


with the right spelling: ['sale', 'new']


### pymongo.errors.WriteError: No array filter found for identifier


In [21]:
try:
    shop.items.update_one({"_id": 1},
                          {"$set": {"lines.$[elem].qty": 9}},
                          array_filters=[{"other.qty": {"$lt": 2}}])
except pymongo.errors.WriteError as error:
    print(failed(error))


WriteError: No array filter found for identifier 'elem' in path 'lines.$[elem].qty'


The path says `$[elem]` and the filter is about `other`. They are the same identifier written twice
and they have to agree, so a rename in one place and not the other produces this.

The error names both halves, which is more than most do, and the fix is to read the path and the
filter next to each other:


In [22]:
fresh_item()
shop.items.update_one({"_id": 1},
                      {"$set": {"lines.$[elem].qty": 9}},
                      array_filters=[{"elem.qty": {"$lt": 2}}])
print("lines now:", item()["lines"])


lines now: [{'sku': 'a', 'qty': 9}, {'sku': 'b', 'qty': 2}]


### No error: find_one_and_update handing back the old document


In [23]:
fresh_item()
print("stock before:", item()["stock"])

returned = shop.items.find_one_and_update({"_id": 1}, {"$inc": {"stock": -1}})
print("what it returned:", returned["stock"])
print("what is now stored:", item()["stock"])
print()
print("a program that trusts the returned value is one behind, forever")


stock before: 4
what it returned: 4
what is now stored: 3

a program that trusts the returned value is one behind, forever


`ReturnDocument.BEFORE` is the default, and the value you get back is the value from before your own
change. Used knowingly it is exactly right for claiming work. Used unknowingly it is an off-by-one
that survives testing, because the number is plausible and only ever wrong by the amount you just
changed it by.


In [24]:
fresh_item()
now = shop.items.find_one_and_update({"_id": 1}, {"$inc": {"stock": -1}},
                                     return_document=ReturnDocument.AFTER)
print("with ReturnDocument.AFTER:", now["stock"], "and stored:", item()["stock"])


with ReturnDocument.AFTER: 3 and stored: 3


In [25]:
shop.items.delete_many({"_id": 1})
shop.stock.drop()
client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- `update_one` takes operators and touches only the fields they name. `replace_one` takes a whole
  document and deletes everything absent from it, keeping only the `_id`.
- PyMongo raises `ValueError` for each shape in the wrong method, which is the only warning you get
  and the reason the mistake is usually caught.
- `$set` and `$unset` add and remove fields. `$inc` and `$mul` do arithmetic on the server, which is
  safe against two writers where read-modify-write is not.
- `$push` always appends, `$addToSet` appends only what is absent, `$pull` removes. The positional
  `$` changes the first element the filter matched, and `array_filters` names elements by condition.
- Operators are case sensitive: `$addToSet`, not `$addtoset`.
- `upsert=True` builds the new document from the filter's **equality** conditions plus the update.
- `find_one_and_update` returns the document from **before** the change unless you pass
  `return_document=ReturnDocument.AFTER`.


## What is next

**Bulk Writes and Transactions** is many changes at once: `bulk_write` with `ordered` either way,
the error whose real message is buried in `.details`, and the write inside a transaction that you
forgot to pass the session to and which therefore survived the abort.


---

&#8592; **Previous:** [Query Operators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/05-query-operators.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
